In [ ]:
import tabula
import faiss
import base64
import pymupdf
import os
import logging
import numpy as np
import warnings
import sys
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython import display
from sentence_transformers import CrossEncoder

logger = logging.getLogger(__name__)
logger.setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [ ]:

pdf_path = r""


if os.path.exists(pdf_path):
    print(f"PDF found successfully: {pdf_path}")
else:
    print(f"PDF not found: {pdf_path}")

PDF found successfully: D:\Prototype1\Rag\1-s2.0-S1877050926007295-main.pdf


In [6]:


def create_directories(base_dir):
    directories = ["images", "text", "tables", "page_images"]
    for directory in directories:
        os.makedirs(os.path.join(base_dir, directory), exist_ok=True)



def process_tables(doc, page_num, base_dir, items):
    try:
        tables = tabula.read_pdf(
            pdf_path,
            pages=page_num + 1,
            multiple_tables=True
        )

        if not tables:
            return

        for table_idx, table in enumerate(tables):
            table_text = "\n".join(
                [" | ".join(map(str, row)) for row in table.values]
            )
   
            table_file_name = os.path.join(
                base_dir,
                "tables",
                f"{os.path.basename(pdf_path)}_table_{page_num}_{table_idx}.txt"
            )

            with open(table_file_name, "w", encoding="utf-8") as f:
                f.write(table_text)

            items.append({
                "page": page_num,
                "type": "table",
                "text": table_text,
                "path": table_file_name
            })

    except Exception as e:
        print(f"Error extracting tables from page {page_num}: {str(e)}")



def process_text_chunks(text, text_splitter, page_num, base_dir, items):
    chunks = text_splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        text_file_name = os.path.join(
            base_dir,
            "text",
            f"{os.path.basename(pdf_path)}_text_{page_num}_{i}.txt"
        )

        with open(text_file_name, "w", encoding="utf-8") as f:
            f.write(chunk)

        items.append({
            "page": page_num,
            "type": "text",
            "text": chunk,
            "path": text_file_name
        })




def process_images(doc, page, page_num, base_dir, items):
    images = page.get_images()

    for idx, image in enumerate(images):
        xref = image[0]

        pix = pymupdf.Pixmap(doc, xref)

        image_name = os.path.join(
            base_dir,
            "images",
            f"{os.path.basename(pdf_path)}_image_{page_num}_{idx}_{xref}.png"
        )

        pix.save(image_name)

        with open(image_name, "rb") as f:
            encoded_image = base64.b64encode(f.read()).decode("utf-8")

        items.append({
            "page": page_num,
            "type": "image",
            "path": image_name,
            "image": encoded_image
        })

def process_page_images(page, page_num, base_dir, items):
    pix = page.get_pixmap()

    page_path = os.path.join(
        base_dir,
        "page_images",
        f"page_{page_num:03d}.png"
    )

    pix.save(page_path)

    with open(page_path, "rb") as f:
        page_image = base64.b64encode(f.read()).decode("utf-8")

    items.append({
        "page": page_num,
        "type": "page",
        "path": page_path,
        "image": page_image
    })

In [7]:

doc = pymupdf.open(pdf_path)

num_pages = len(doc)

base_dir = "data"

create_directories(base_dir)


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    length_function=len
)


items = []


for page_num in tqdm(range(num_pages), desc="Processing PDF pages"):

    page = doc[page_num]

   
    text = page.get_text()

   
    process_tables(
        doc,
        page_num,
        base_dir,
        items
    )

  
    process_text_chunks(
        text,
        text_splitter,
        page_num,
        base_dir,
        items
    )

   
    process_images(
        doc,
        page,
        page_num,
        base_dir,
        items
    )

   
    process_page_images(
        page,
        page_num,
        base_dir,
        items
    )

print(f"PDF processing completed: {num_pages} pages processed.")
print(f"Total extracted items: {len(items)}")

Processing PDF pages:   0%|          | 0/9 [00:00<?, ?it/s]Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'
Processing PDF pages: 100%|██████████| 9/9 [01:29<00:00,  9.97s/it]

PDF processing completed: 9 pages processed.
Total extracted items: 73


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:

embedding_vector_dimension = 384

item_counts = {
    "text": sum(1 for item in items if item["type"] == "text"),
    "table": sum(1 for item in items if item["type"] == "table"),
    "image": sum(1 for item in items if item["type"] == "image"),
    "page": sum(1 for item in items if item["type"] == "page")
}

counters = dict.fromkeys(item_counts.keys(), 0)
with tqdm(
    total=len(items),
    desc="Generating embeddings",
    bar_format=(
        "{l_bar}{bar}| {n_fmt}/{total_fmt} "
        "[{elapsed}<{remaining}, {rate_fmt}{postfix}]"
    )
) as pbar:

    for item in items:

        item_type = item["type"]
        counters[item_type] += 1

        if item_type in ["text", "table"]:

           
            embedding = embedding_model.encode(
                item["text"],
                convert_to_numpy=True,
                normalize_embeddings=True
            )

            item["embedding"] = embedding.astype("float32")

        else:

            item["embedding"] = None

       
        pbar.set_postfix_str(
            f"Text: {counters['text']}/{item_counts['text']}, "
            f"Table: {counters['table']}/{item_counts['table']}"
        )

        pbar.update(1)

print("Embedding generation completed.")

Generating embeddings: 100%|██████████| 73/73 [00:23<00:00,  3.05it/s, Text: 59/59, Table: 0/0]

Embedding generation completed.


In [11]:

embedded_items = [
    item for item in items
    if item["embedding"] is not None
]

all_embeddings = np.array(
    [item["embedding"] for item in embedded_items],
    dtype=np.float32
)


index = faiss.IndexFlatL2(embedding_vector_dimension)


index.reset()

index.add(all_embeddings)

print(f"FAISS index created successfully.")
print(f"Number of vectors indexed: {index.ntotal}")
print(f"Embedding dimension: {index.d}")

FAISS index created successfully.
Number of vectors indexed: 59
Embedding dimension: 384


In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI


load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. "
        "Check that .env contains GROQ_API_KEY."
    )


client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq client configured successfully.")

Groq client configured successfully.


In [13]:
def invoke_groq_multimodal(prompt, matched_items):
    """
    Generate an answer using Groq from retrieved PDF context.
    """

    system_message = """
You are a helpful assistant for question answering.

The context provided below was retrieved from a PDF document.

Answer the user's question using only the provided context.

Do not invent information.

If the answer cannot be found in the provided context,
say that the information was not found in the document.
"""

   
    context_parts = []

    for item in matched_items:
        if item["type"] in ["text", "table"]:
            context_parts.append(
                f"[Page {item['page']}]\n{item['text']}"
            )

    context = "\n\n".join(context_parts)

    
    user_message = f"""
Retrieved document context:

{context}

Question:
{prompt}
"""

   
    response = client.responses.create(
        model="openai/gpt-oss-20b",
        input=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    return response.output_text

In [14]:
def ask_pdf(question, k=5):

   
    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)


    k = min(k, index.ntotal)

    distances, result = index.search(
        query_embedding.reshape(1, -1),
        k
    )

   
    matched_items = [
        {
            key: value
            for key, value in embedded_items[idx].items()
            if key != "embedding"
        }
        for idx in result[0]
        if idx != -1
    ]

    print(f"Retrieved {len(matched_items)} relevant chunks.")

   
    response = invoke_groq_multimodal(
        question,
        matched_items
    )

    return response

In [17]:
query = input("Ask a question about the PDF: ")

response = ask_pdf(query)

display.Markdown(response)

Retrieved 5 relevant chunks.


The adoption of a more expressive embedding model (MiniLM‑L12‑v2) together with an instruction‑tuned language model **resulted in improvements in both fragment diversity and answer precision**. This demonstrates the importance of aligning the retrieval pipeline with the generative model to produce coherent, document‑grounded responses.